In [1]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_11.pickle

In [4]:
%%RecordEvent
%%time
### cell 11 ###

country_order = df['first_country'].value_counts()[:11].index

# Use GPU‐friendly pivot instead of CPU‐bound value_counts + unstack
counts = df.groupby(['first_country', 'type']).size().reset_index(name='count')
data_q2q3 = (
    counts
    .pivot(index='first_country', columns='type', values='count')
    .fillna(0)
    .loc[country_order]
)

# compute total and ratios
data_q2q3['sum'] = data_q2q3.sum(axis=1)
data_q2q3_ratio = (
    data_q2q3[['Movie', 'TV Show']]
    .div(data_q2q3['sum'], axis=0)
    .sort_values(by='Movie', ascending=False)
    .iloc[::-1]
)

CPU times: user 284 ms, sys: 81.6 ms, total: 365 ms
Wall time: 400 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_11_try_1.pickle